In [29]:
from pathlib import Path
from collections import Counter
import json
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re

In [30]:
ROOT_DIR = Path.cwd().parent.parent
data_file_path = ROOT_DIR / "data" / "fashion_products_dataset.json" 

with open(data_file_path, "r") as f:
    products = json.load(f)

print(f"Total products loaded：{len(products):,}")

Total products loaded：28,080


In [31]:
attributes = ["pid", "title", "description", "brand", "category", "sub_category", "product_details", "seller", "out_of_stock", 
"selling_price", "discount", "actual_price", "average_rating", "url"]

for attribute in attributes:
    missing_count = sum(1 for product in products if product.get(attribute) == "" or product.get(attribute) is None)
    print(f"{attribute}: {missing_count:,}, missing percentage: {missing_count / len(products) * 100:.2f}%")

pid: 0, missing percentage: 0.00%
title: 0, missing percentage: 0.00%
description: 11,150, missing percentage: 39.71%
brand: 2,009, missing percentage: 7.15%
category: 0, missing percentage: 0.00%
sub_category: 0, missing percentage: 0.00%
product_details: 0, missing percentage: 0.00%
seller: 1,643, missing percentage: 5.85%
out_of_stock: 0, missing percentage: 0.00%
selling_price: 2, missing percentage: 0.01%
discount: 855, missing percentage: 3.04%
actual_price: 777, missing percentage: 2.77%
average_rating: 2,261, missing percentage: 8.05%
url: 0, missing percentage: 0.00%


In [32]:
nltk.download("stopwords", quiet=True, raise_on_error=True)

stop_words = stopwords.words("english")
print(f"First 10 stop words: {stop_words[:10]}")

stop_words = set(stop_words)
stemmer = PorterStemmer()

def preprocess_text(text):
    if not text:
        return []
    
    text = text.lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    
    tokens_removed_stopwords = [token for token in tokens if token not in stop_words]
    tokens_stemmed = [stemmer.stem(token) for token in tokens_removed_stopwords]

    return tokens_stemmed

product = products[0]
print(type(product["product_details"]), product["product_details"])
print(type(product["product_details"][0]), product["product_details"][0])

print(f"Original product title: {product["title"]}")
print(f"Preprocessed product title: {preprocess_text(product["title"])}")

print(f"Original product description: {product["description"]}")
print(f"Preprocessed product description: {preprocess_text(product["description"])}")

First 10 stop words: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']
<class 'list'> [{'Style Code': '1005COMBO2'}, {'Closure': 'Elastic'}, {'Pockets': 'Side Pockets'}, {'Fabric': 'Cotton Blend'}, {'Pattern': 'Solid'}, {'Color': 'Multicolor'}]
<class 'dict'> {'Style Code': '1005COMBO2'}
Original product title: Solid Women Multicolor Track Pants
Preprocessed product title: ['solid', 'women', 'multicolor', 'track', 'pant']
Original product description: Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India
Preprocessed product description: ['yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'look', 'design', 'comfort', 'skin', 'friendli', 'fabric', 'itch', 'free', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india']


In [33]:
""" EDA here """

' EDA here '

In [34]:
attributes = ["category", "sub_category", "brand", "product_details", "seller"]

for attribute in attributes:
    print(f"{attribute}: {products[0].get(attribute)}")


def normalize_attribute(text):
    if not text:
        return ""
    
    return " ".join(text.lower().split())

def product_details_to_text(product_details):
    attribute_texts = []

    for detail in product_details:
        for key, value in detail.items():
            if value is not None:
                attribute_texts.append(f"{key} {value}")

    return " ".join(attribute_texts)

metadata_attributes = ["category", "sub_category", "brand", "seller"]

preprocessed_products = []

for product in products:
    preprocessed_product = product.copy()
    preprocessed_product["title_tokens"] = preprocess_text(product["title"])
    preprocessed_product["description_tokens"] = preprocess_text(product["description"])
    
    for attribute in metadata_attributes:
        preprocessed_product[f"{attribute}_normalized"] = normalize_attribute(product[attribute])

    detail_text = product_details_to_text(product["product_details"])
    preprocessed_product["product_details_tokens"] = preprocess_text(detail_text)
    
    preprocessed_products.append(preprocessed_product)

preprocessed_product = preprocessed_products[0]

print(f"Preprocessed products: {len(preprocessed_products):,}")

print(f"Original product title: {preprocessed_product["title"]}")
print(f"Preprocessed product title: {preprocessed_product["title_tokens"]}")

print(f"Original product description: {preprocessed_product["description"]}")
print(f"Preprocessed product description: {preprocessed_product["description_tokens"]}")

for attribute in metadata_attributes:
    print(f"Original product {attribute}: {preprocessed_product[attribute]}")
    print(f"Normalized product {attribute}: {preprocessed_product[f"{attribute}_normalized"]}")

print(f"Original product details: {preprocessed_product["product_details"]}")
print(f"Preprocessed product details: {preprocessed_product["product_details_tokens"]}")

category: Clothing and Accessories
sub_category: Bottomwear
brand: York
product_details: [{'Style Code': '1005COMBO2'}, {'Closure': 'Elastic'}, {'Pockets': 'Side Pockets'}, {'Fabric': 'Cotton Blend'}, {'Pattern': 'Solid'}, {'Color': 'Multicolor'}]
seller: Shyam Enterprises
Preprocessed products: 28,080
Original product title: Solid Women Multicolor Track Pants
Preprocessed product title: ['solid', 'women', 'multicolor', 'track', 'pant']
Original product description: Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India
Preprocessed product description: ['yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'look', 'design', 'comfort', 'skin', 'friendli', 'fabric', 'itch', 'free', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india']
Original product category: Clothing and Accessories
Normalized product categ

In [ ]:
def parse_price(value):
    if value is None:
        return None

    text = value.strip().replace(",", "")

    try:
        price = float(text)
    except ValueError:
        return None

    return price

def parse_discount(value):
    value = value.replace("%", "").replace("off", "").strip()

    discount = parse_price(value)

    if discount is None or not (0 <= discount <= 100):
        return None

    return discount

numerical_attributes = ["selling_price", "actual_price", "average_rating"]

for product in preprocessed_products:
    for attribute in numerical_attributes:
        product[f"{attribute}_numeric"] = parse_price(product[attribute])

    product["discount_numeric"] = parse_discount(product["discount"])

preprocessed_product = preprocessed_products[0]

for attribute in numerical_attributes + ["discount"]:
    print(f"{attribute}: {preprocessed_product[attribute]!r}") # !r let original string be printed with quotes
    print(f"{attribute}_numeric: {preprocessed_product[f'{attribute}_numeric']}")


selling_price: '921'
selling_price_numeric: 921.0
actual_price: '2,999'
actual_price_numeric: 2999.0
average_rating: '3.9'
average_rating_numeric: 3.9
discount: '69% off'
discount_numeric: 69.0


In [42]:
output_file_path = ROOT_DIR / "data" / "preprocessed_fashion_products_dataset.json"

with open(output_file_path, "w", encoding="utf-8") as file:
    json.dump(preprocessed_products, file, ensure_ascii=False, indent=4, allow_nan=False)

print(f"Finished => Preprocessed products saved to: {output_file_path}")

Finished => Preprocessed products saved to: /Users/xia/Documents/University/Recuperació de la Informació i Anàlisi de la Web/Final Project/irwa-search-engine/data/preprocessed_fashion_products_dataset.json
